# 03 — Baseline LSTM + Grid Search

perbaikan baseline dari kak steven:
- Arsitektur 2 layer LSTM + Dense(25) + Dense(1) (sama dengan Gambar 5 skripsi kak steven).
- Grid: `learning_rate ∈ {1e-3, 1e-2}`, `optimizer ∈ {adam, rmsprop}`, `units ∈ {16, 32, 64}`, `dropout ∈ {0, 0.1, 0.2}`.
- **Perbaikan di caps vs skripsi:** scaler hanya fit pada train, EarlyStopping aktif, metrik dihitung pada skala asli µg/m³.

Output: CSV berisi metrik tiap kombinasi per stasiun + model `.keras` terbaik.

In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd

from src import config as C
from src.data_loader import load_all_stations
from src.preprocessing import build_pipeline
from src.tuning_grid import grid_search, DEFAULT_GRID

In [3]:
FEATURES = C.WEATHER_COLS + [C.AOD_COL, C.TARGET]
STATIONS_TO_RUN = C.HEALTHY_STATIONS  # skip jakarta_gbk yang missing 2 tahun
EPOCHS = 100
PATIENCE = 15

In [4]:
dfs = load_all_stations(reindex=True)
all_results = []
best_per_station = {}

for s in STATIONS_TO_RUN:
    print(f'\n=== {s} ===')
    data = build_pipeline(dfs[s], FEATURES, smooth_aod=False)
    df_res, best = grid_search(data, grid=DEFAULT_GRID, epochs=EPOCHS, patience=PATIENCE, verbose=1)
    df_res['station'] = s
    df_res.to_csv(C.METRICS_DIR / f'03_grid_{s}.csv', index=False)
    best['model'].save(C.MODEL_DIR / f'03_grid_{s}.keras')
    best_per_station[s] = {'params': best['params'], 'R2': best['R2']}
    all_results.append(df_res)
    print(f'   best R²={best["R2"]:.4f} | {best["params"]}')


=== us_embassy_1 ===
[1/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.0}
[2/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.1}
[3/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.2}
[4/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 32, 'dropout_rate': 0.0}
[5/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 32, 'dropout_rate': 0.1}
[6/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 32, 'dropout_rate': 0.2}
[7/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 64, 'dropout_rate': 0.0}
[8/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 64, 'dropout_rate': 0.1}
[9/36] {'learning_rate': 0.001, 'optimizer': 'adam', 'lstm_units': 64, 'dropout_rate': 0.2}
[10/36] {'learning_rate': 0.001, 'optimizer': 'rmsprop', 'lstm_units': 16, 'dropout_rate': 0.0}
[11/36] {'learning_rate': 0.001, 'optimizer': 'rmsprop

In [5]:
# Rangkuman best per stasiun
summary = pd.DataFrame([
    {'station': s, 'R2': v['R2'], **v['params']}
    for s, v in best_per_station.items()
]).sort_values('R2', ascending=False)
summary.to_csv(C.METRICS_DIR / '03_grid_best_summary.csv', index=False)
summary

,station,R2,learning_rate,optimizer,lstm_units,dropout_rate
3,kelapa_gading,0.634426,0.001,adam,16,0.1
2,bundaran_hi,0.596756,0.001,adam,16,0.0
6,kebun_jeruk,0.591850,0.001,adam,16,0.0
4,jagakarsa,0.488738,0.001,adam,16,0.2
1,us_embassy_2,0.415823,0.001,adam,16,0.1
5,lubang_buaya,0.017091,0.001,adam,32,0.0
0,us_embassy_1,-0.217306,0.001,adam,32,0.2


In [6]:
# Gabungkan semua hasil grid search untuk analisis
df_all = pd.concat(all_results, ignore_index=True)
df_all.to_csv(C.METRICS_DIR / '03_grid_all_results.csv', index=False)
df_all.head()

,learning_rate,optimizer,lstm_units,dropout_rate,val_R2,val_MSE,val_RMSE,val_MAE,test_R2,test_MSE,test_RMSE,test_MAE,epochs_run,best_train_loss,best_val_loss,station
0,0.001,adam,32,0.2,0.498231,206.464771,14.368882,11.115862,-0.217306,5.412901e+02,23.265641,18.736941,80,0.011203,0.018583,us_embassy_1
1,0.001,adam,64,0.1,0.483451,212.546183,14.578964,11.506001,-19632.910172,8.730461e+06,2954.735308,1562.642660,90,0.009661,0.019130,us_embassy_1
2,0.001,adam,16,0.2,0.472453,217.071597,14.733350,12.262111,-0.239908,5.513403e+02,23.480637,17.797092,100,0.012143,0.019537,us_embassy_1
3,0.001,adam,16,0.0,0.468616,218.650350,14.786830,12.045126,-197.660868,8.833701e+04,297.215426,225.433564,100,0.010187,0.019680,us_embassy_1
4,0.001,adam,64,0.2,0.452018,225.480174,15.015997,12.179307,-456.376311,2.033780e+05,450.974531,287.284358,77,0.011986,0.020294,us_embassy_1
